# Step 1 — Data Acquisition

Pull 7 years of adjusted close prices for **KO** (Coca-Cola) and **PEP** (PepsiCo),
validate the data, and cache to Parquet.

**Why adjusted closes?** Both companies pay quarterly dividends. Without adjustment,
ex-dividend dates look like sudden price drops — noise that would corrupt every
downstream analysis. `auto_adjust=True` handles splits and dividends simultaneously.

**Why Parquet cache?** We never re-download inside an analysis loop. Pull once,
validate once, read the cache forever. Parquet is columnar, compressed, and
preserves datetime types — much better than CSV for this.

In [1]:
import sys
from pathlib import Path

# Make the pairs_trading module importable from the notebooks directory
sys.path.insert(0, str(Path.cwd().parent))

from pairs_trading.data import load_or_fetch, fetch_earnings_dates, describe_prices

## Parameters

7 years of daily data gives ~1,760 observations — enough statistical power for
cointegration tests (power is ~95% at 1,000+ obs) while keeping the window
modern enough to reflect the current competitive relationship between the two companies.

In [2]:
TICKERS = ["KO", "PEP"]
START   = "2018-01-01"
END     = "2025-01-01"

## Fetch and validate

First run downloads from Yahoo Finance and saves the Parquet cache.
Every subsequent run loads from disk — no network call needed.

In [3]:
prices = load_or_fetch(TICKERS, START, END, name="ko_pep_7yr")

Loading from cache: c:\Users\delosreyes.i\Downloads\Quant Claude Project\Pair-Regression-Testing\pairs_trading\data\ko_pep_7yr.parquet


## Inspect the data

Check: ~1,760 rows, no NaNs, sensible price ranges.
If count < 1,750 or any column has NaN, investigate before moving on.

In [4]:
describe_prices(prices)


--- Price data summary ---
Date range : 2018-01-02 to 2024-12-31
Trading days: 1761
Columns    : ['KO', 'PEP']

Latest prices:
                   KO         PEP
2024-12-27  60.246071  145.713257
2024-12-30  59.840893  144.598190
2024-12-31  60.062771  144.922211

Basic stats:
                KO          PEP
count  1761.000000  1761.000000
mean     48.082274   129.450005
std       8.916003    27.719638
min      31.304207    75.162003
25%      40.051491   108.814026
50%      47.623512   128.156540
75%      55.717758   154.805695
max      69.427246   177.249725


In [5]:
prices.head()

,KO,PEP
2018-01-02,35.250359,91.602783
2018-01-03,35.172955,91.362282
2018-01-04,35.668346,91.812302
2018-01-05,35.660599,92.076088
2018-01-08,35.606419,91.548485


In [6]:
prices.tail()

,KO,PEP
2024-12-24,60.622307,145.637024
2024-12-26,60.361832,145.284363
2024-12-27,60.246071,145.713257
2024-12-30,59.840893,144.598190
2024-12-31,60.062771,144.922211


## Earnings dates

We cache these alongside prices. They're used for two things later:
1. **Visualization** — mark earnings on spread plots so we can see if price
   dislocations coincide with announcements
2. **Backtest exclusion** — optionally skip holding positions through earnings
   (event risk can swamp the mean-reversion signal)

Note the large PEP misses in 2025 — those could stress the spread significantly.

In [7]:
for ticker in TICKERS:
    try:
        earnings = fetch_earnings_dates(ticker)
        print(f"\n{ticker} earnings (most recent 8):")
        display(earnings.head(8))
    except Exception as e:
        print(f"{ticker} earnings fetch failed: {e}")


KO earnings (most recent 8):


,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-07-21 08:00:00-04:00,0.93,NaN,NaN
2026-04-28 07:00:00-04:00,0.81,0.86,5.88
2026-02-10 06:00:00-05:00,0.56,0.58,2.69
2025-10-21 06:00:00-04:00,0.78,0.82,5.32
2025-07-22 07:00:00-04:00,0.84,0.87,3.93
2025-04-29 06:00:00-04:00,0.72,0.73,1.95
2025-02-11 06:00:00-05:00,0.52,0.55,6.38
2024-10-23 06:00:00-04:00,0.75,0.77,3.18



PEP earnings (most recent 8):


,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-07-16 08:00:00-04:00,2.23,NaN,NaN
2026-04-16 06:00:00-04:00,1.55,1.61,3.75
2026-02-03 06:00:00-05:00,2.21,1.85,-16.14
2025-10-09 06:00:00-04:00,2.27,1.90,-16.30
2025-07-17 06:00:00-04:00,2.03,0.92,-54.68
2025-04-24 06:00:00-04:00,1.49,1.48,-0.77
2025-02-04 06:00:00-05:00,1.94,1.96,0.90
2024-10-08 06:00:00-04:00,2.29,2.31,0.74


## Sanity check summary

Expected results for a clean 7-year KO/PEP pull:

| Check | Expected | Status |
|---|---|---|
| Trading days | ~1,760 | ✓ |
| NaN values | 0 | ✓ |
| Date gaps > 7 days | 0 | ✓ |
| KO price range | $30–$70 | ✓ |
| PEP price range | $75–$180 | ✓ |

All checks pass. Ready for Step 2 — exploratory visualization.